In [10]:
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
from lightgbm import LGBMClassifier
import warnings
from sklearn.utils import shuffle
import matplotlib.pyplot as plt
import joblib, os
from lightgbm import LGBMRanker
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [2]:
df = pd.read_csv("data/master_dataset.csv")

In [3]:
leakage_cols = ["omdb_oscar_wins", "omdb_awards", "days_to_ceremony"]


In [4]:
# Features absolutos

abs_features = [
    "imdb_rating", "rt_score", "metacritic", "tmdb_vote_avg",
    "tmdb_popularity", "log_imdb_votes", "critic_composite",
    "budget_m", "revenue_m", "roi",
    "runtime_min", "is_q4_release", "is_english", "n_nominees_year",
    "total_precursor_wins", "total_precursor_noms",
    "BAFTA_best_film_won", "GG_drama_won", "GG_comedy_won",
    "CCA_best_picture_won", "PGA_best_picture_won",
    "genre_drama", "genre_biography", "genre_history",
    "genre_romance", "genre_thriller", "genre_war",
]

In [5]:
# Relativos al año

rel_features = [f"{f}_pct_year" for f in [
    "imdb_rating", "rt_score", "metacritic", "tmdb_vote_avg",
    "tmdb_popularity", "budget_m", "revenue_m",
    "total_precursor_wins", "critic_composite"
]] + [
    "imdb_rating_is_max", "rt_score_is_max", "metacritic_is_max",
    "total_precursor_wins_is_max", "is_precursor_leader",
    "critic_composite_is_max",
]

In [6]:
all_features = [f for f in abs_features + rel_features
                if f in df.columns and f not in leakage_cols]

In [ ]:
train_years = list(range(2005, 2019))
val_years   = list(range(2019, 2022))
test_years  = list(range(2022, 2026))

df_train    = df[df["ceremony_year"].isin(train_years)]
df_val      = df[df["ceremony_year"].isin(val_years)]
df_test     = df[df["ceremony_year"].isin(test_years)]
df_trainval = pd.concat([df_train, df_val])

X_train = df_train[all_features].fillna(-999)
y_train = df_train["won_best_picture"]

In [8]:
# ── Preparar grupos (requerido por LambdaMART) ────────────────────────
# group = cantidad de películas por año, en orden
def get_groups(data, years):
    return [len(data[data["ceremony_year"] == y]) for y in sorted(years)]

# Ordenar por año
df_train_s    = df_train.sort_values("ceremony_year")
df_val_s      = df_val.sort_values("ceremony_year")
df_trainval_s = df_trainval.sort_values("ceremony_year")

train_groups    = get_groups(df_train_s,    sorted(train_years))
val_groups      = get_groups(df_val_s,      sorted(val_years))
trainval_groups = get_groups(df_trainval_s, sorted(train_years + val_years))

X_tr = df_train_s[all_features].fillna(-999)
y_tr = df_train_s["won_best_picture"]
X_vl = df_val_s[all_features].fillna(-999)
y_vl = df_val_s["won_best_picture"]

In [11]:
# ── Tuning con Optuna ─────────────────────────────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_ranker(trial):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 100, 500),
        "learning_rate"    : trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "max_depth"        : trial.suggest_int("max_depth", 2, 6),
        "num_leaves"       : trial.suggest_int("num_leaves", 6, 40),
        "min_child_samples": trial.suggest_int("min_child_samples", 2, 15),
        "subsample"        : trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha"        : trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),
        "reg_lambda"       : trial.suggest_float("reg_lambda", 1e-4, 1.0, log=True),
        "random_state"     : 42,
        "verbose"          : -1,
    }
    ranker = LGBMRanker(objective="lambdarank", **params)
    ranker.fit(X_tr, y_tr, group=train_groups,
               eval_set=[(X_vl, y_vl)], eval_group=[val_groups])
    
    # Evaluar percentil en val
    scores = []
    for year in val_years:
        year_df    = df_val_s[df_val_s["ceremony_year"] == year]
        scores_r   = ranker.predict(year_df[all_features].fillna(-999))
        winner_s   = scores_r[year_df["won_best_picture"].values == 1][0]
        scores.append((scores_r < winner_s).mean())
    return np.mean(scores)

study_ranker = optuna.create_study(direction="maximize",
                                   sampler=optuna.samplers.TPESampler(seed=42))
study_ranker.optimize(objective_ranker, n_trials=100, show_progress_bar=True)

best_params_ranker = study_ranker.best_params
best_params_ranker.update({"random_state": 42, "verbose": -1})
print(f"\nMejor val score ranker: {study_ranker.best_value:.3f}")

  0%|          | 0/100 [00:00<?, ?it/s]


Mejor val score ranker: 0.838


In [12]:
# ── Reentrenar con train+val ──────────────────────────────────────────
X_tv = df_trainval_s[all_features].fillna(-999)
y_tv = df_trainval_s["won_best_picture"]

ranker_final = LGBMRanker(objective="lambdarank", **best_params_ranker)
ranker_final.fit(X_tv, y_tv, group=trainval_groups)

# ── Test ──────────────────────────────────────────────────────────────
print("\n── LightGBM Ranker — Test Set (2022-2025) ───────────────────")
ranker_results = []
for year in test_years:
    year_df     = df_test[df_test["ceremony_year"] == year].copy()
    scores_r    = ranker_final.predict(year_df[all_features].fillna(-999))
    scores_norm = (scores_r - scores_r.min()) / (scores_r.max() - scores_r.min() + 1e-9)
    scores_norm = scores_norm / scores_norm.sum()
    year_df["prob_ranker"] = scores_norm

    pred  = year_df.sort_values("prob_ranker", ascending=False).iloc[0]["nominated_title"]
    real  = year_df[year_df["won_best_picture"] == 1]["nominated_title"].values[0]
    wprob = year_df[year_df["won_best_picture"] == 1]["prob_ranker"].values[0]
    ok    = int(pred == real)

    print(f"{year} | Real: {real:<40} Pred: {pred:<40} {wprob:.1%} {'✅' if ok else '❌'}")
    ranker_results.append({"year": year, "real": real, "pred": pred,
                            "correct": ok, "winner_prob": wprob, "model": "Ranker"})

ranker_df = pd.DataFrame(ranker_results)
print(f"\nAccuracy Ranker: {ranker_df['correct'].mean():.1%}")
joblib.dump(ranker_final, "models/ranker_oscar.pkl")


── LightGBM Ranker — Test Set (2022-2025) ───────────────────
2022 | Real: CODA                                     Pred: CODA                                     26.3% ✅
2023 | Real: Everything Everywhere All at Once        Pred: Everything Everywhere All at Once        27.9% ✅
2024 | Real: Oppenheimer                              Pred: Oppenheimer                              34.9% ✅
2025 | Real: Anora                                    Pred: Anora                                    29.0% ✅

Accuracy Ranker: 100.0%


['models/ranker_oscar.pkl']